# Week 4: LLM and ML Benchmarking

This notebook shows a simple way to test prompt versions across three LLM endpoints.

Goal: choose robust prompts and document failure modes for each model.

## Step 1: Imports and Constants

What this does: imports libraries for data processing, prompt parsing, and model calls.

Why it matters: this keeps each later step focused on logic, not setup.

Principle: set up your toolbox first so experiments are repeatable and easy to debug.

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

openai = __import__('openai')
OpenAI = openai.OpenAI

MODEL_ENDPOINTS = [
    {'label': 'phi-3.5-mini-instruct', 'model_name': 'Phi-3.5-mini-instruct', 'host': 'localhost', 'port': 8000},
    {'label': 'phi-mini-moe-instruct', 'model_name': 'Phi-mini-MoE-instruct', 'host': 'localhost', 'port': 8001},
    {'label': 'gemma-3-12b-it', 'model_name': 'gemma-3-12b-it', 'host': 'localhost', 'port': 8002},
]
API_KEY = 'EMPTY'

# ── Paths ──────────────────────────────────────────────────────────
ROOT = Path('..').resolve()
DATA_CSV = ROOT/'week3_artifacts/FTES_cleaned_1sec_1min_resample.csv'
ML_CSV = ROOT/'week3_artifacts/ml_results.csv'
SPLIT_CSV = ROOT/'week3_artifacts/splits_indexed.csv'
PROMPT_DIR = ROOT / "Prompts"
PROMPT_DIR.mkdir(exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────
ID_COL = "timestamp"
SPLIT_COL = "split"
TARGET_REF_COL = "delta_T_above_T0_TN"
HORIZONS = [15, 60, 240, 1440]
TARGET_COLS = ["target_15", "target_60", "target_240", "target_1440"]
ML_ID_COL = "Test_Idx"
ML_COLS = ["Multi-XGB_+15m", "Multi-XGB_+60m", "Multi-XGB_+240m", "Multi-XGB_+1440m"]

BASE_FEATURES = [
    "cumulative_heat_input", "elapsed_injection_min", "net_flow_rolling_6h",
    "TC_INT_delta", "T_gradient_INT_TC", "days_since_injection",
    "hour_sin", "hour_cos", "delta_T_above_T0_TN",
]

FEATURES_FULL = [
    "cumulative_heat_input", "elapsed_injection_min", "net_flow_rolling_6h",
    "TC_INT_delta", "T_gradient_INT_TC", "days_since_injection",
    "hour_sin", "hour_cos", "delta_T_above_T0_TN",
    "target_lag_15", "flow_lag_15", "target_lag_30", "flow_lag_30",
    "target_lag_60", "flow_lag_60",
]

RESPONSE_SCHEMA = '{"target_15": <float>, "target_60": <float>, "target_240": <float>, "target_1440": <float>, "confidence": <0-1>}'

print(f"ROOT:       {ROOT}")
print(f"DATA_CSV:   {DATA_CSV}  (exists={DATA_CSV.exists()})")
print(f"PROMPT_DIR: {PROMPT_DIR}")
print(f"Features:   {len(FEATURES_FULL)}")


ROOT:       /home/jupyter-xue.li/ByteMe_week_4
DATA_CSV:   /home/jupyter-xue.li/ByteMe_week_4/week3_artifacts/FTES_cleaned_1sec_1min_resample.csv  (exists=True)
PROMPT_DIR: /home/jupyter-xue.li/ByteMe_week_4/Prompts
Features:   15


## Step 2: Load Week 3 test data

What this does: loads the Week 3 held-out split and defines the allowed label set.

Why it matters: prompt comparison is only meaningful when all versions are tested on the same target and data.

Principle: control your dataset before comparing methods, otherwise comparisons are not fair.

In [2]:
# Load only the columns we need
df = pd.read_csv(DATA_CSV)[[ID_COL]+BASE_FEATURES].copy()
df_ml = pd.read_csv(ML_CSV,index_col=ML_ID_COL)[ML_COLS].copy()
df_split = pd.read_csv(SPLIT_CSV).copy()

# Lag features
for lag in [15, 30, 60]:
    df[f"target_lag_{lag}"] = df[TARGET_REF_COL].shift(lag)
    df[f"flow_lag_{lag}"] = df["net_flow_rolling_6h"].shift(lag)

# Forward targets
for h in HORIZONS:
    df[f"target_{h}"] = df[TARGET_REF_COL].shift(-h)

# Split
idx_train = df_split[SPLIT_COL] == "train"
idx_test = df_split[SPLIT_COL] == "test"
df_train = df.loc[idx_train].copy()
df_test = df.loc[idx_test].copy()

# Append ML predictions
df_test = pd.merge(df_test, df_ml,left_index=True, right_index=True)

print(f"Train rows: {len(df_train):,}   Test rows: {len(df_test):,}")
display(df_test.head())

Train rows: 71,924   Test rows: 24,235


,timestamp,cumulative_heat_input,elapsed_injection_min,net_flow_rolling_6h,TC_INT_delta,T_gradient_INT_TC,days_since_injection,hour_sin,hour_cos,delta_T_above_T0_TN,...,target_lag_60,flow_lag_60,target_15,target_60,target_240,target_1440,Multi-XGB_+15m,Multi-XGB_+60m,Multi-XGB_+240m,Multi-XGB_+1440m
72001,2025-02-01 20:01:00,4.239934e+08,72001.0,1.485943,0.768405,-5.345621,50.000694,-0.863836,0.503774,5.607457,...,5.584420,1.485944,5.595543,5.570608,5.606484,5.576771,5.604058,5.605496,5.606494,5.635816
72002,2025-02-01 20:02:00,4.239971e+08,72002.0,1.485940,0.151337,-5.331847,50.001389,-0.861629,0.507538,5.598087,...,5.592866,1.485954,5.576046,5.591359,5.572344,5.554104,5.575201,5.576639,5.577637,5.612080
72003,2025-02-01 20:03:00,4.240009e+08,72003.0,1.485936,-0.175045,-5.325941,50.002083,-0.859406,0.511293,5.568759,...,5.612416,1.485928,5.604481,5.603992,5.600141,5.569208,5.584404,5.585842,5.586841,5.620599
72004,2025-02-01 20:04:00,4.240046e+08,72004.0,1.485961,-0.910542,-5.348824,50.002778,-0.857167,0.515038,5.605122,...,5.577321,1.485904,5.582582,5.570827,5.607526,5.580339,5.603888,5.605326,5.606325,5.635647
72005,2025-02-01 20:05:00,4.240084e+08,72005.0,1.485971,0.228062,-5.353266,50.003472,-0.854912,0.518773,5.582435,...,5.607153,1.485920,5.586700,5.602448,5.563109,5.552715,5.565057,5.566495,5.567493,5.608487


## Step 3: Define prompt versions

What this does: creates alternative prompt templates (baseline and few-shot) and one parser for all outputs.

Why it matters: you can test formatting and accuracy differences while keeping evaluation logic constant.

Principle: change one variable at a time. Here, prompt text changes while parser and scoring stay fixed.

In [3]:
# ═══════════════════════════════════════════════════════════════════
# Sampling
# ═══════════════════════════════════════════════════════════════════
def sample_evenly_spaced(df, num_examples, features=None):
    """Sample evenly-spaced rows across time, dropping rows with NaN."""
    cols = list(features or [c for c in df.columns if c != SPLIT_COL])
    cols_needed = list(set(cols) | set(TARGET_COLS))
    df_clean = df[[c for c in cols_needed if c in df.columns]].dropna()
    if len(df_clean) < num_examples:
        raise ValueError(f"Only {len(df_clean)} complete rows, need {num_examples}")
    idxs = np.linspace(0, len(df_clean) - 1, num_examples, dtype=int)
    return df_clean.iloc[idxs]


# ═══════════════════════════════════════════════════════════════════
# Formatters
# ═══════════════════════════════════════════════════════════════════
def format_verbose(row, features, targets=TARGET_COLS, precision=4):
    """Named key=value pairs in natural language."""
    feat_str = ", ".join(f"{c}={row[c]:.{precision}f}" for c in features)
    tgt = {c: round(row[c], precision) for c in targets}
    tgt["confidence"] = 1.0
    prompt = f"Given features: {feat_str}\nPredict temperature delta and confidence."
    completion = json.dumps(tgt)
    return {"prompt": prompt, "completion": completion}


def format_compact(row, features, targets=TARGET_COLS, precision=2):
    """Compact input array, RESPONSE_SCHEMA output."""
    inp = [round(row[c], precision) for c in features]
    tgt = {c: round(row[c], precision) for c in targets}
    tgt["confidence"] = 1.0
    return {"i": inp, "completion": json.dumps(tgt)}


FORMATTERS = {"verbose": format_verbose, "compact": format_compact}


# ═══════════════════════════════════════════════════════════════════
# System message
# ═══════════════════════════════════════════════════════════════════
def build_system_message(style, features=FEATURES_FULL):
    """Build the system instruction for the LLM."""
    msg = (
        "You are a geothermal temperature forecasting model.\n"
        f"Respond ONLY with valid JSON matching this schema:\n{RESPONSE_SCHEMA}\n"
        "target_15 = predicted delta_T at 15 min, target_60 = at 1 h, "
        "target_240 = at 4 h, target_1440 = at 24 h.\n"
        "confidence = your estimated probability that the prediction is "
        "within 10% of the true value (0-1)."
    )
    if style == "compact":
        msg += f"\nFeature order: {features}\nOutput order: {TARGET_COLS + ['confidence']}"
    return msg


# ═══════════════════════════════════════════════════════════════════
# Zero-shot prompt builder
# ═══════════════════════════════════════════════════════════════════
def build_zero_shot_prompt(test_row, style="verbose", features=FEATURES_FULL, precision=None):
    """Build a zero-shot prompt (system + single query, no examples)."""
    prec = precision or (4 if style == "verbose" else 2)
    fmt_fn = FORMATTERS[style]
    system = build_system_message(style, features)

    test_ex = fmt_fn(test_row, features, TARGET_COLS, prec)
    if style == "verbose":
        query = test_ex["prompt"]
    else:
        query = json.dumps(test_ex["i"])

    return {"system": system, "user": f"Q: {query}\nA:", "parse_hint": "json"}


# ═══════════════════════════════════════════════════════════════════
# Few-shot generation & prompt builder
# ═══════════════════════════════════════════════════════════════════
def generate_few_shot(df_train, num_examples=5, features=FEATURES_FULL,
                      style="verbose", precision=None):
    """Sample training examples and format them as few-shot examples."""
    fmt_fn = FORMATTERS[style]
    prec = precision or (4 if style == "verbose" else 2)
    samples = sample_evenly_spaced(df_train, num_examples, features)
    examples = [fmt_fn(row, features, TARGET_COLS, prec) for _, row in samples.iterrows()]
    return {
        "meta": {
            "style": style,
            "features": features,
            "targets": TARGET_COLS,
            "response_schema": RESPONSE_SCHEMA,
            "num_examples": num_examples,
        },
        "examples": examples,
    }


def build_few_shot_prompt(few_shot, test_row, features=None, style=None, precision=None):
    """Assemble system + N example Q/A pairs + test query."""
    meta = few_shot["meta"]
    feat_list = features or meta["features"]
    st = style or meta["style"]
    prec = precision or (4 if st == "verbose" else 2)
    fmt_fn = FORMATTERS[st]

    system = build_system_message(st, feat_list)

    # Few-shot examples block
    lines = []
    for ex in few_shot["examples"]:
        if st == "verbose":
            lines.append(f"Q: {ex['prompt']}\nA: {ex['completion']}")
        else:
            lines.append(f"Q: {json.dumps(ex['i'])}\nA: {ex['completion']}")

    # Test query
    test_ex = fmt_fn(test_row, feat_list, TARGET_COLS, prec)
    if st == "verbose":
        query = test_ex["prompt"]
    else:
        query = json.dumps(test_ex["i"])

    user = "\n\n".join(lines) + f"\n\nQ: {query}\nA:"
    return {"system": system, "user": user, "parse_hint": "json"}


# ═══════════════════════════════════════════════════════════════════
# Save / load helpers
# ═══════════════════════════════════════════════════════════════════
def save_prompts(data, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
    print(f"Saved → {output_path}  ({output_path.stat().st_size / 1024:.1f} KB)")
    
# ═══════════════════════════════════════════════════════════════════
# Parse LLM response
# ═══════════════════════════════════════════════════════════════════
def parse_response(raw_text, valid_range=None):
    """Extract predictions + confidence from LLM response string.

    Args:
        raw_text:     Raw LLM output string.
        valid_range:  Optional (min, max) tuple; predictions outside this range
                      are flagged as invalid.

    Returns:
        (pred_dict, confidence, is_valid, error_reason)
        pred_dict:    dict with all TARGET_COLS as keys, or None
        confidence:   float in [0, 1] or np.nan
        is_valid:     bool
        error_reason: None when valid, else one of
                      'empty_response', 'invalid_json', 'missing_targets',
                      'non_numeric', 'out_of_range'
    """
    if not isinstance(raw_text, str) or not raw_text.strip():
        return None, np.nan, False, 'empty_response'

    text = raw_text.strip()

    # --- Try to extract JSON object or array from the response ---
    m = re.search(r'\{.*\}|\[.*\]', text, flags=re.DOTALL)
    candidate = m.group(0) if m else text

    try:
        payload = json.loads(candidate)
    except Exception:
        return None, np.nan, False, 'invalid_json'

    # --- Normalise compact array [t15, t60, t240, t1440, conf] → dict ---
    if isinstance(payload, list):
        if len(payload) < len(TARGET_COLS):
            return None, np.nan, False, 'missing_targets'
        payload = {t: payload[i] for i, t in enumerate(TARGET_COLS)}
        payload['confidence'] = payload.get('confidence', payload[len(TARGET_COLS)]) if len(payload) > len(TARGET_COLS) else np.nan

    # --- Extract targets ---
    values = {}
    for t in TARGET_COLS:
        v = payload.get(t)
        if v is None:
            return None, np.nan, False, 'missing_targets'
        try:
            values[t] = float(v)
        except (TypeError, ValueError):
            return None, np.nan, False, 'non_numeric'

    # --- Extract confidence ---
    conf = payload.get('confidence', np.nan)
    try:
        conf = float(conf)
    except (TypeError, ValueError):
        conf = np.nan

    pred = dict(values)

    # --- Optional range check ---
    if valid_range is not None:
        lo, hi = valid_range
        if any(not (lo <= pred[t] <= hi) for t in TARGET_COLS):
            return pred, conf, False, 'out_of_range'

    return pred, conf, True, None


print("All core functions defined ✓")


All core functions defined ✓


In [4]:

# ── Configuration ──────────────────────────────────────────────────
FEW_SHOT_STYLES = ["verbose", "compact"]
FEW_SHOT_COUNTS = {
    "verbose": [1, 5, 10],          # verbose prompts are long; keep shot count lower
    "compact": [1, 5, 10, 20],  # compact prompts are short; can afford more shots
}

# Pre-generate few-shot example sets for each (style, count) combination
few_shot_sets = {}
for style in FEW_SHOT_STYLES:
    for n in FEW_SHOT_COUNTS[style]:
        tag = f"{style}_{n}"
        fs = generate_few_shot(df_train, num_examples=n, features=FEATURES_FULL, style=style)
        few_shot_sets[tag] = fs
        print(f"  {tag}: {n} examples, style={style}")

print(f"\nGenerated {len(few_shot_sets)} few-shot example sets")

# Also save the few-shot example sets themselves (reusable without re-sampling)
for tag, fs in few_shot_sets.items():
    out_path = PROMPT_DIR / f"examples_{tag}.json"
    save_prompts(fs, out_path)


  verbose_1: 1 examples, style=verbose
  verbose_5: 5 examples, style=verbose
  verbose_10: 10 examples, style=verbose
  compact_1: 1 examples, style=compact
  compact_5: 5 examples, style=compact
  compact_10: 10 examples, style=compact
  compact_20: 20 examples, style=compact

Generated 7 few-shot example sets
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/examples_verbose_1.json  (1.3 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/examples_verbose_5.json  (3.6 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/examples_verbose_10.json  (6.5 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/examples_compact_1.json  (1.1 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/examples_compact_5.json  (2.6 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/examples_compact_10.json  (4.5 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/examples_compact_20.json  (8.2 KB)


## Step 4: Evaluate prompt versions on a small subset

What this does: runs each prompt version on the same subset and computes parse success and accuracy.

Why it matters: small-scale testing is faster and helps pick a strong prompt before full benchmarking.

Principle: iterate quickly on a representative subset, then scale to the full test set once stable.

WARNING: Nested for loops + LLM queries = death by boredom and kernel crashes from time outs. In your actual testing keep subsets small and/or query one model at a time.

In [ ]:

# Auto-built from FEW_SHOT_STYLES / FEW_SHOT_COUNTS — edit those to change what's tested
VERSIONS = (
    [(style, 'zeroshot') for style in FEW_SHOT_STYLES]
    + [(style, str(n)) for style in FEW_SHOT_STYLES for n in FEW_SHOT_COUNTS[style]]
)

N_TEST_SAMPLES = 5
VALID_RANGE = (-10, 10)      # flag predictions outside this range

clients = {
    cfg['label']: OpenAI(
        api_key=API_KEY,
        base_url=f"http://{cfg['host']}:{cfg['port']}/v1"
    )
    for cfg in MODEL_ENDPOINTS
}

def query_llm(prompt_dict, endpoint_cfg, temperature=0.0, seed=0):
    """Send system + user prompt to an OpenAI-compatible endpoint."""
    client = clients[endpoint_cfg['label']]
    response = client.chat.completions.create(
        model=endpoint_cfg['model_name'],
        messages=[
            {'role': 'system', 'content': prompt_dict['system']},
            {'role': 'user',   'content': prompt_dict['user']},
        ],
        temperature=temperature,
        seed=seed,
    )
    return response.choices[0].message.content

def build_prompt(row,style,fewshot=None):
    if fewshot is not None:
        prompt_dict = build_few_shot_prompt(fewshot, row)
    else:
        prompt_dict = build_zero_shot_prompt(row, style=style)
    return prompt_dict

def pred_llm(input_row, endpoint, version):
    style = version.split('_')[0]
    rows = version.split('_')[1]
    flag_fewshots = True
    if rows == 'zeroshot':
        few_shot = None
        prompt_file = f'Prompts/prompt_template_{endpoint["label"]}.txt'
    else:
        fs_path = PROMPT_DIR / f'examples_{version}.json'
        if not fs_path.exists():
            raise ValueError(f"Error processing {version}: {fs_path} not found")
        few_shot = json.loads(fs_path.read_text(encoding="utf-8"))
        prompt_file = f'Prompts/examples_{version}.json'
    prompt_dict = build_prompt(input_row,style=style,fewshot=few_shot)
    raw = query_llm(prompt_dict, endpoint, temperature=0.0, seed=0)
    pred, conf, parse_ok, parse_error = parse_response(raw, valid_range=VALID_RANGE)

    record = {
        'timestamp':      input_row[ID_COL],
        'model_label':    endpoint['label'],
        'model_name':     endpoint['model_name'],
        'endpoint_port':  endpoint['port'],
        'version':        version,
        'prompt_file':    prompt_file,
        'parse_ok':       parse_ok,
        'parse_error':    parse_error,
        'confidence':     conf,
        'raw_response':   raw,
    }
    # Ground truth
    for t in TARGET_COLS:
        record[f'gt_{t}'] = input_row[t]
    # ML predictions
    for t in ML_COLS:
        record[f'ml_{t}'] = input_row[t]
    # Predictions
    if pred is not None:
        for t in TARGET_COLS:
            record[f'pred_{t}'] = pred[t]
            record[f'err2_{t}'] = (pred[t] - input_row[t])**2
    else:
        for t in TARGET_COLS:
            record[f'pred_{t}'] = np.nan
            record[f'err2_{t}'] = np.nan
    return(record)


In [6]:
# Sample test rows (evenly spaced, no NaN)
test_subset = sample_evenly_spaced(df_test, N_TEST_SAMPLES)

records = []
for style, rows in VERSIONS:
    tag = f"{style}_{rows}" 
    for endpoint_cfg in MODEL_ENDPOINTS:
        for idx, row in test_subset.iterrows():
            record = pred_llm(row, endpoint_cfg, tag)
            records.append(record)
            print(f"  {tag} | {endpoint_cfg['label']} | ts={idx} | ok={record['parse_ok']}")

results = pd.DataFrame(records)

summary = results.groupby(
    ['model_label', 'model_name', 'endpoint_port', 'version'],
    as_index=False,
).agg(
    parse_success_rate=('parse_ok', 'mean'),
    mse_15=('err2_target_15', 'mean'),
    mse_60=('err2_target_60', 'mean'),
    mse_240=('err2_target_240', 'mean'),
    mse_1440=('err2_target_1440', 'mean'),
    avg_confidence=('confidence', 'mean'),
    rows=('parse_ok', 'size'),
)
summary['mse_overall'] = summary[['mse_15', 'mse_60', 'mse_240', 'mse_1440']].mean(axis=1)
summary = summary.sort_values(
    ['model_label', 'parse_success_rate', 'mse_overall'],
    ascending=[True, False, True],
)

# Save
OUT_DIR = ROOT / 'Results'
results.to_csv(OUT_DIR / 'llm_inference_results.csv', index=False)
summary.to_csv(OUT_DIR / 'llm_inference_summary.csv', index=False)
display(summary)


  verbose_zeroshot | phi-3.5-mini-instruct | ts=72001 | ok=True
  verbose_zeroshot | phi-3.5-mini-instruct | ts=76590 | ok=True
  verbose_zeroshot | phi-3.5-mini-instruct | ts=84118 | ok=True
  verbose_zeroshot | phi-3.5-mini-instruct | ts=88707 | ok=True
  verbose_zeroshot | phi-3.5-mini-instruct | ts=96235 | ok=True
  verbose_zeroshot | phi-mini-moe-instruct | ts=72001 | ok=True
  verbose_zeroshot | phi-mini-moe-instruct | ts=76590 | ok=True
  verbose_zeroshot | phi-mini-moe-instruct | ts=84118 | ok=False
  verbose_zeroshot | phi-mini-moe-instruct | ts=88707 | ok=True
  verbose_zeroshot | phi-mini-moe-instruct | ts=96235 | ok=True
  verbose_zeroshot | gemma-3-12b-it | ts=72001 | ok=True
  verbose_zeroshot | gemma-3-12b-it | ts=76590 | ok=True
  verbose_zeroshot | gemma-3-12b-it | ts=84118 | ok=True
  verbose_zeroshot | gemma-3-12b-it | ts=88707 | ok=True
  verbose_zeroshot | gemma-3-12b-it | ts=96235 | ok=True
  compact_zeroshot | phi-3.5-mini-instruct | ts=72001 | ok=True
  compact_

,model_label,model_name,endpoint_port,version,parse_success_rate,mse_15,mse_60,mse_240,mse_1440,avg_confidence,rows,mse_overall
6,gemma-3-12b-it,gemma-3-12b-it,8002,verbose_10,1.0,0.000056,0.000116,0.000112,0.000298,1.000,5,0.000145
2,gemma-3-12b-it,gemma-3-12b-it,8002,compact_20,1.0,0.000032,0.000329,0.000077,0.000190,1.000,5,0.000157
3,gemma-3-12b-it,gemma-3-12b-it,8002,compact_5,1.0,0.000074,0.000231,0.000095,0.000354,1.000,5,0.000189
1,gemma-3-12b-it,gemma-3-12b-it,8002,compact_10,1.0,0.000075,0.000138,0.000175,0.000421,1.000,5,0.000202
7,gemma-3-12b-it,gemma-3-12b-it,8002,verbose_5,1.0,0.000048,0.000117,0.000117,0.000564,1.000,5,0.000212
5,gemma-3-12b-it,gemma-3-12b-it,8002,verbose_1,1.0,0.000392,0.002902,0.009046,0.101399,0.970,5,0.028435
4,gemma-3-12b-it,gemma-3-12b-it,8002,compact_zeroshot,1.0,0.004844,0.025321,0.057346,0.117986,0.850,5,0.051374
0,gemma-3-12b-it,gemma-3-12b-it,8002,compact_1,1.0,0.018519,0.061028,0.149417,0.866527,0.950,5,0.273873
8,gemma-3-12b-it,gemma-3-12b-it,8002,verbose_zeroshot,1.0,33.013461,32.797909,32.830639,32.705111,0.850,5,32.836780
15,phi-3.5-mini-instruct,Phi-3.5-mini-instruct,8000,verbose_10,1.0,0.000128,0.000120,0.000148,0.000339,1.000,5,0.000184


## Step 5: Inspect failures

What this does: shows failed parses with raw responses and error labels.

Why it matters: this evidence is what you use for the required Week 4 error analysis.

Principle: error analysis should be example-driven, not only metric-driven.

In [7]:
failures = results[~results['parse_ok']].copy()
display(failures[['model_label','version', 'raw_response', 'parse_error']])
print(f'Total failures: {len(failures)} out of {len(results)}')

,model_label,version,raw_response,parse_error
7,phi-mini-moe-instruct,verbose_zeroshot,"{\n ""target_15"": 5.8162,\n ""target_60"": 5.7...",missing_targets
120,phi-3.5-mini-instruct,compact_20,"{""target_15"": 5.58, ""target_60"": 5.57, ""targe...",invalid_json
123,phi-3.5-mini-instruct,compact_20,"{""target_15"": 5.86, ""target_60"": 5.86, ""targe...",invalid_json


Total failures: 3 out of 135


## Step 6: Save final prompt files

What this does: writes your selected prompt template and few-shot examples to the submission folder.

Why it matters: prompt artifacts make your workflow auditable and reproducible for judges.

Principle: version your prompts like code. Prompt files are part of the experiment definition.

In [8]:
best_by_model = (
    summary.sort_values(['model_label', 'parse_success_rate', 'mse_overall'], ascending=[True, False, True])
    .groupby('model_label', as_index=False)
    .first()
    [['model_label', 'model_name', 'endpoint_port', 'version']]
 )
display(best_by_model)

# Save one canonical template + few-shot file for submission compatibility.
overall_best_version = summary.sort_values(['parse_success_rate', 'mse_overall'], ascending=[False, True]).iloc[0]['version']
print(f'Overall best prompt version: {overall_best_version}')

style = overall_best_version.split('_')[0]
fewshot_flag =  'zeroshot' not in overall_best_version
demo_row = sample_evenly_spaced(df_test, 1, FEATURES_FULL).iloc[0]
final_prompt_example = build_prompt(
    row = demo_row,
    style = style,
    fewshot = json.loads((PROMPT_DIR / f'examples_{overall_best_version}.json').read_text(encoding="utf-8")) if fewshot_flag else None
)
prompt_template_path = PROMPT_DIR / 'prompt_template.txt'
save_prompts(final_prompt_example,prompt_template_path)

# Also save model-specific prompt choices.
for _, row in best_by_model.iterrows():
    style = row.version.split('_')[0]
    fewshot_flag =  'zeroshot' not in row.version
    model_prompt = build_prompt(
        row = demo_row,
        style = style,
        fewshot = json.loads((PROMPT_DIR / f'examples_{row.version}.json').read_text(encoding="utf-8")) if fewshot_flag else None
)
    model_prompt_path = PROMPT_DIR / f"prompt_template_{row['model_label']}.txt"
    save_prompts(model_prompt,model_prompt_path)


,model_label,model_name,endpoint_port,version
0,gemma-3-12b-it,gemma-3-12b-it,8002,verbose_10
1,phi-3.5-mini-instruct,Phi-3.5-mini-instruct,8000,verbose_10
2,phi-mini-moe-instruct,Phi-mini-MoE-instruct,8001,compact_20


Overall best prompt version: compact_20
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/prompt_template.txt  (5.2 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/prompt_template_gemma-3-12b-it.txt  (6.3 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/prompt_template_phi-3.5-mini-instruct.txt  (6.3 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/prompt_template_phi-mini-moe-instruct.txt  (5.2 KB)


## Step 7: Full Benchmark

What this does: loops over all configured model endpoints and a larger subset of test rows, storing raw responses and parsed predictions.

Why it matters: this produces side-by-side model outputs from a single consistent pipeline.

Principle: hold the dataset constant while varying models with prompting strategy to compare best performing versions.

In [9]:
N_TEST_LARGE = 100
test_subset_large = sample_evenly_spaced(df_test, N_TEST_LARGE)
# display(test_subset_large.head())

def find_by_label(label: str) -> dict | None:
    return next((ep for ep in MODEL_ENDPOINTS if ep['label'] == label), None)

def calculate_mse(actual, predicted):
    mse = ((actual - predicted) ** 2).mean()
    return mse

def calculate_r2(actual, predicted):
    ss_res = ((actual - predicted) ** 2).sum()        # Residual Sum of Squares
    ss_tot = ((actual - actual.mean()) ** 2).sum()    # Total Sum of Squares    
    r2 = 1 - (ss_res / ss_tot)
    return r2

def evaluate(df_results,label):
    results = []
    for _, h in enumerate(HORIZONS):
        actual = df_results[f'gt_target_{h}']
        ml = df_results[f'ml_Multi-XGB_+{h}m']
        llm = df_results[f'pred_target_{h}']
        mse_llm = calculate_mse(actual, llm)
        r2_llm = calculate_r2(actual, llm)
        mse_ml = calculate_mse(actual, ml)
        r2_ml = calculate_r2(actual, ml)
        print(f"[{label}] Horizon +{h}m -> MSE_LLM: {mse_llm:.4f} | MSE_ML: {mse_ml:.4f} | R2_LLM: {r2_llm:.4f} | R2_ML: {r2_ml:.4f}")
        results.append({'Model': label, 'Horizon': h, 
                        'MSE_LLM': mse_llm, 'MSE_ML': mse_ml, 
                        'R2_LLM': r2_llm, 'R2_ML': r2_ml})
    return results

performance_metrics = []
for _,row in best_by_model.iterrows():
#     print(row)
    endpoint_config = find_by_label(row.model_label)
    records = []
    for i,rec in test_subset_large.iterrows():
        record = pred_llm(rec, endpoint_config, row.version)
        records.append(record)
    results = pd.DataFrame(records)
    results.to_csv(OUT_DIR/f"llm_benchmark_{row['model_label']}.csv")
    performance = evaluate(results,row.model_label)
    performance_metrics += performance

df_performance_metrics = pd.DataFrame(performance_metrics)
df_performance_metrics.to_csv(OUT_DIR/f"llm_benchmark_performance.csv")
display(df_performance_metrics)
    
    

[gemma-3-12b-it] Horizon +15m -> MSE_LLM: 0.0002 | MSE_ML: 0.0067 | R2_LLM: 0.9853 | R2_ML: 0.4854
[gemma-3-12b-it] Horizon +60m -> MSE_LLM: 0.0002 | MSE_ML: 0.0060 | R2_LLM: 0.9856 | R2_ML: 0.5756
[gemma-3-12b-it] Horizon +240m -> MSE_LLM: 0.0004 | MSE_ML: 0.0051 | R2_LLM: 0.9736 | R2_ML: 0.6313
[gemma-3-12b-it] Horizon +1440m -> MSE_LLM: 0.0017 | MSE_ML: 0.0050 | R2_LLM: 0.8742 | R2_ML: 0.6163
[phi-3.5-mini-instruct] Horizon +15m -> MSE_LLM: 0.0003 | MSE_ML: 0.0067 | R2_LLM: 0.9807 | R2_ML: 0.4854
[phi-3.5-mini-instruct] Horizon +60m -> MSE_LLM: 0.0002 | MSE_ML: 0.0060 | R2_LLM: 0.9860 | R2_ML: 0.5756
[phi-3.5-mini-instruct] Horizon +240m -> MSE_LLM: 0.0003 | MSE_ML: 0.0051 | R2_LLM: 0.9806 | R2_ML: 0.6313
[phi-3.5-mini-instruct] Horizon +1440m -> MSE_LLM: 0.0015 | MSE_ML: 0.0050 | R2_LLM: 0.8886 | R2_ML: 0.6163
[phi-mini-moe-instruct] Horizon +15m -> MSE_LLM: 0.0002 | MSE_ML: 0.0067 | R2_LLM: 0.9812 | R2_ML: 0.4854
[phi-mini-moe-instruct] Horizon +60m -> MSE_LLM: 0.0002 | MSE_ML: 0.

,Model,Horizon,MSE_LLM,MSE_ML,R2_LLM,R2_ML
0,gemma-3-12b-it,15,0.000193,0.006733,0.985270,0.485427
1,gemma-3-12b-it,60,0.000203,0.005992,0.985649,0.575629
2,gemma-3-12b-it,240,0.000366,0.005103,0.973567,0.631258
3,gemma-3-12b-it,1440,0.001654,0.005045,0.874240,0.616294
4,phi-3.5-mini-instruct,15,0.000253,0.006733,0.980683,0.485427
5,phi-3.5-mini-instruct,60,0.000197,0.005992,0.986027,0.575629
6,phi-3.5-mini-instruct,240,0.000269,0.005103,0.980573,0.631258
7,phi-3.5-mini-instruct,1440,0.001464,0.005045,0.888645,0.616294
8,phi-mini-moe-instruct,15,0.000247,0.006733,0.981157,0.485427
9,phi-mini-moe-instruct,60,0.000244,0.005992,0.982738,0.575629


## Step 8: Agentic Prompt Auto-Design

What this does: each LLM designs its own prompt for the forecasting task from scratch, tests it, reviews parse failures and accuracy, then iterates. The LLM has full creative freedom — it chooses prompting style, whether to include examples, how many, and what format.

Why it matters: automated prompt engineering can discover model-specific strategies humans wouldn't try. Each model may prefer a different style.

Principle: close the loop — let the model that will be tested also shape the prompt it responds to, then measure whether self-designed prompts beat manually crafted ones.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Agentic prompt auto-design helpers
# ═══════════════════════════════════════════════════════════════════
MAX_PROMPT_TOKENS = 3500   # leave room for LLM response within 4096 context

FEATURE_DESCRIPTIONS = (
    "cumulative_heat_input: total thermal energy injected (MJ), "
    "elapsed_injection_min: minutes since injection started, "
    "net_flow_rolling_6h: 6-hour rolling average net flow (L/min), "
    "TC_INT_delta: injection well interval temperature change (°C), "
    "T_gradient_INT_TC: temperature gradient in injection well interval (°C), "
    "days_since_injection: days elapsed since first injection, "
    "hour_sin/hour_cos: cyclical hour-of-day encoding, "
    "delta_T_above_T0_TN: current temperature delta above baseline (°C), "
    "target_lag_15/30/60: delta_T values at 15/30/60 min ago, "
    "flow_lag_15/30/60: net_flow values at 15/30/60 min ago"
)


def _approx_tokens(text):
    """Rough token count (1 token ≈ 4 chars)."""
    return len(text) // 4


def _format_training_pool(df_pool, features, precision=2):
    """Format training rows as compact JSON lines for the meta-prompt."""
    lines = []
    for _, row in df_pool.iterrows():
        entry = {
            "input": [round(row[f], precision) for f in features],
            "output": {t: round(row[t], precision) for t in TARGET_COLS},
        }
        lines.append(json.dumps(entry))
    return "\n".join(lines)


def build_meta_prompt(features, response_schema, training_pool_str,
                      failure_examples=None, prev_metrics=None, round_num=0):
    """Ask the LLM to freely design a prompting strategy for the forecasting task.
    
    Returns a prompt dict {"system": ..., "user": ...} for the meta-design call.
    """
    system = (
        "You are a prompt-engineering expert. Your job is to design the best possible "
        "prompt (system message + user message template) for a geothermal temperature "
        "forecasting task. You have full creative freedom: choose any prompting style, "
        "decide whether to include examples, how many, and in what format.\n\n"
        "Respond ONLY with valid JSON matching this exact schema:\n"
        '{"system_prompt": "<the system message you design>", '
        '"user_template": "<user message template, MUST contain the literal placeholder {features_json}>", '
        '"strategy_notes": "<brief explanation of your design choices>"}\n\n'
        "IMPORTANT: The user_template MUST contain the exact string {features_json} "
        "as a placeholder where the test row data will be inserted."
    )

    user_parts = [
        "TASK: Predict geothermal temperature deltas at 4 future horizons from sensor readings.",
        f"FEATURES ({len(features)} inputs): {FEATURE_DESCRIPTIONS}",
        f"FEATURE NAMES (in order): {features}",
        f"REQUIRED OUTPUT JSON SCHEMA: {response_schema}",
        f"CONTEXT WINDOW LIMIT: The final rendered prompt must fit in ~{MAX_PROMPT_TOKENS} tokens.",
        "",
        f"LABELLED TRAINING DATA ({training_pool_str.count(chr(10))+1} rows, input→output):",
        training_pool_str,
        "",
        "Design a complete prompt strategy. You decide:",
        "- System message content and tone",
        "- Whether to include few-shot examples (and how many from the pool above)",
        "- Input/output formatting style (verbose, compact, tabular, etc.)",
        "- Any other prompting techniques (chain-of-thought, role-play, etc.)",
        "",
        "The model receiving your prompt MUST respond with ONLY valid JSON matching the schema above.",
    ]

    if round_num > 0 and failure_examples:
        user_parts.append("")
        user_parts.append(f"── ITERATION FEEDBACK (round {round_num}) ──")
        if prev_metrics:
            user_parts.append(
                f"Previous round: parse_success={prev_metrics['parse_rate']:.0%}, "
                f"mse={prev_metrics['mse']:.4f}"
            )
        user_parts.append("Failed responses from your previous prompt design:")
        for i, fail in enumerate(failure_examples[:5]):  # cap at 5
            raw_snip = str(fail.get('raw_response', ''))[:200]
            user_parts.append(
                f"  Failure {i+1} [{fail.get('parse_error','?')}]: {raw_snip}"
            )
        user_parts.append("")
        user_parts.append("Fix the issues above. Redesign your prompt to improve parse reliability and accuracy.")

    user = "\n".join(user_parts)
    return {"system": system, "user": user}


def parse_meta_response(raw_text):
    """Extract the LLM-designed prompt from the meta-design response.
    
    Returns (system_prompt, user_template, strategy_notes, is_valid, error).
    """
    if not isinstance(raw_text, str) or not raw_text.strip():
        return None, None, None, False, "empty_response"

    m = re.search(r'\{.*\}', raw_text, flags=re.DOTALL)
    if not m:
        return None, None, None, False, "no_json_found"

    try:
        payload = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None, None, None, False, "invalid_json"

    system_prompt = payload.get("system_prompt")
    user_template = payload.get("user_template")
    strategy_notes = payload.get("strategy_notes", "")

    if not system_prompt or not isinstance(system_prompt, str):
        return None, None, None, False, "missing_system_prompt"
    if not user_template or not isinstance(user_template, str):
        return None, None, None, False, "missing_user_template"
    if "{features_json}" not in user_template:
        return None, None, None, False, "missing_placeholder"

    return system_prompt, user_template, strategy_notes, True, None


def build_auto_prompt(system_prompt, user_template, test_row, features, precision=3):
    """Render an auto-designed prompt template for a concrete test row."""
    features_data = {f: round(float(test_row[f]), precision) for f in features}
    features_json = json.dumps(features_data)
    user = user_template.replace("{features_json}", features_json)

    total_text = system_prompt + user
    tok_est = _approx_tokens(total_text)
    if tok_est > MAX_PROMPT_TOKENS:
        print(f"  ⚠ Rendered prompt ~{tok_est} tokens (limit {MAX_PROMPT_TOKENS})")

    return {"system": system_prompt, "user": user, "parse_hint": "json"}


def pred_auto(input_row, endpoint, system_prompt, user_template, features=FEATURES_FULL):
    """Run a single prediction using an auto-designed prompt, returning a record dict."""
    prompt_dict = build_auto_prompt(system_prompt, user_template, input_row, features)
    raw = query_llm(prompt_dict, endpoint, temperature=0.0, seed=0)
    pred, conf, parse_ok, parse_error = parse_response(raw, valid_range=VALID_RANGE)

    record = {
        'timestamp':      input_row[ID_COL],
        'model_label':    endpoint['label'],
        'model_name':     endpoint['model_name'],
        'endpoint_port':  endpoint['port'],
        'version':        'auto_designed',
        'prompt_file':    f'Prompts/auto_prompt_{endpoint["label"]}.json',
        'parse_ok':       parse_ok,
        'parse_error':    parse_error,
        'confidence':     conf,
        'raw_response':   raw,
    }
    for t in TARGET_COLS:
        record[f'gt_{t}'] = input_row[t]
    for t in ML_COLS:
        record[f'ml_{t}'] = input_row[t]
    if pred is not None:
        for t in TARGET_COLS:
            record[f'pred_{t}'] = pred[t]
            record[f'err2_{t}'] = (pred[t] - input_row[t])**2
    else:
        for t in TARGET_COLS:
            record[f'pred_{t}'] = np.nan
            record[f'err2_{t}'] = np.nan
    return record


def run_agentic_loop(endpoint_cfg, df_train, test_subset,
                     features=FEATURES_FULL, response_schema=RESPONSE_SCHEMA,
                     max_rounds=3, n_pool=10, valid_range=(-10, 10)):
    """Run the full agentic prompt-design loop for one model.
    
    Returns (best_design, best_records, round_history) where:
        best_design:  dict with system_prompt, user_template, strategy_notes
        best_records: list of prediction record dicts from the best round
        round_history: list of per-round summaries
    """
    label = endpoint_cfg['label']
    print(f"\n{'='*60}")
    print(f"Agentic loop: {label}  (max_rounds={max_rounds})")
    print(f"{'='*60}")

    # Build training pool for meta-prompt
    pool = sample_evenly_spaced(df_train, n_pool, features)
    pool_str = _format_training_pool(pool, features)

    best_design = None
    best_records = []
    best_parse_rate = -1.0
    best_mse = float('inf')
    round_history = []
    prev_failures = None
    prev_metrics = None

    for rnd in range(max_rounds):
        print(f"\n── Round {rnd} ──")

        # 1. Build meta-prompt and ask LLM to design a prompt
        meta_prompt = build_meta_prompt(
            features, response_schema, pool_str,
            failure_examples=prev_failures,
            prev_metrics=prev_metrics,
            round_num=rnd,
        )
        meta_tok = _approx_tokens(meta_prompt['system'] + meta_prompt['user'])
        print(f"  Meta-prompt: ~{meta_tok} tokens")

        try:
            raw_design = query_llm(meta_prompt, endpoint_cfg, temperature=0.0, seed=rnd)
        except Exception as e:
            print(f"  ✗ Meta-query failed: {e}")
            round_history.append({'round': rnd, 'status': 'meta_query_failed', 'error': str(e)})
            continue

        # 2. Parse the designed prompt
        system_p, user_t, notes, ok, err = parse_meta_response(raw_design)
        if not ok:
            print(f"  ✗ Meta-parse failed: {err}")
            print(f"    Raw (first 300 chars): {raw_design[:300]}")
            round_history.append({'round': rnd, 'status': 'meta_parse_failed', 'error': err,
                                  'raw_design': raw_design[:500]})
            # Feed this failure back as context for next round
            prev_failures = [{'raw_response': raw_design, 'parse_error': f'meta_design_{err}'}]
            continue

        print(f"  ✓ Strategy: {notes[:120]}")

        # 3. Test the designed prompt on test subset
        records = []
        for _, row in test_subset.iterrows():
            record = pred_auto(row, endpoint_cfg, system_p, user_t, features)
            records.append(record)

        # 4. Evaluate
        df_rnd = pd.DataFrame(records)
        parse_rate = df_rnd['parse_ok'].mean()
        mse_cols = [f'err2_{t}' for t in TARGET_COLS]
        mse = df_rnd[mse_cols].mean().mean() if parse_rate > 0 else float('inf')

        print(f"  Parse rate: {parse_rate:.0%}  |  MSE: {mse:.4f}")

        round_info = {
            'round': rnd, 'status': 'ok',
            'parse_rate': parse_rate, 'mse': mse,
            'strategy_notes': notes,
            'system_prompt': system_p,
            'user_template': user_t,
        }
        round_history.append(round_info)

        # 5. Track best
        if (parse_rate > best_parse_rate) or \
           (parse_rate == best_parse_rate and mse < best_mse):
            best_parse_rate = parse_rate
            best_mse = mse
            best_design = {
                'system_prompt': system_p,
                'user_template': user_t,
                'strategy_notes': notes,
                'round': rnd,
                'parse_rate': parse_rate,
                'mse': mse,
            }
            best_records = records
            print(f"  ★ New best (round {rnd})")

        # 6. Prepare feedback for next round
        prev_failures = [r for r in records if not r['parse_ok']]
        prev_metrics = {'parse_rate': parse_rate, 'mse': mse}

    if best_design is not None:
        print(f"\nBest design for {label}: round {best_design['round']}, "
              f"parse={best_design['parse_rate']:.0%}, mse={best_design['mse']:.4f}")
    else:
        print(f"\n⚠ No valid prompt design produced across {max_rounds} rounds for {label}")

    return best_design, best_records, round_history


print("Agentic helper functions defined ✓")


Agentic helper functions defined ✓


In [11]:
# ── Agentic loop configuration ─────────────────────────────────────
AGENTIC_MAX_ROUNDS = 10       # max prompt-design iterations per model
AGENTIC_N_TEST = 5           # test samples per round (keep small for speed)
AGENTIC_N_POOL = 10          # labelled training rows provided to agent

agentic_test_subset = sample_evenly_spaced(df_test, AGENTIC_N_TEST)

# Run the agentic loop for each model
agentic_results = {}
for endpoint_cfg in MODEL_ENDPOINTS:
    label = endpoint_cfg['label']
    best_design, best_records, history = run_agentic_loop(
        endpoint_cfg=endpoint_cfg,
        df_train=df_train,
        test_subset=agentic_test_subset,
        features=FEATURES_FULL,
        response_schema=RESPONSE_SCHEMA,
        max_rounds=AGENTIC_MAX_ROUNDS,
        n_pool=AGENTIC_N_POOL,
    )
    agentic_results[label] = {
        'design': best_design,
        'records': best_records,
        'history': history,
    }

# Quick summary
for label, res in agentic_results.items():
    d = res['design']
    print(f"\n{label}:")
    print(f"  Strategy: {d['strategy_notes'][:150]}")
    print(f"  Parse: {d['parse_rate']:.0%}  MSE: {d['mse']:.4f}  (designed in round {d['round']})")



Agentic loop: phi-3.5-mini-instruct  (max_rounds=10)

── Round 0 ──
  Meta-prompt: ~1098 tokens
  ✓ Strategy: The system message includes a clear task description and a few-shot example to provide context and demonstrate the expec
  Parse rate: 80%  |  MSE: 0.0009
  ★ New best (round 0)

── Round 1 ──
  Meta-prompt: ~1210 tokens
  ✗ Meta-parse failed: invalid_json
    Raw (first 300 chars):  ```json
{
  "system_prompt": "Given the sensor readings for geothermal temperature forecasting, predict the temperature deltas at 15, 60, 240, and 1440 minutes ahead. Use the provided features and their corresponding values to train a model. Ensure the output is in the specified JSON format with co

── Round 2 ──
  Meta-prompt: ~1213 tokens
  ✓ Strategy: The system message is designed to be clear and instructive, providing context and examples to guide the model. The user 
  Parse rate: 0%  |  MSE: inf

── Round 3 ──
  Meta-prompt: ~1439 tokens
  ✓ Strategy: The system message is designed to be cl

  ✓ Strategy: This prompt utilizes a direct, task-oriented approach. The system prompt establishes a clear role for the model (geother
  Parse rate: 100%  |  MSE: 0.0007

── Round 8 ──
  Meta-prompt: ~1098 tokens
  ✓ Strategy: This prompt utilizes a direct, task-oriented approach. The system prompt establishes a clear role for the model (geother
  Parse rate: 100%  |  MSE: 0.0007

── Round 9 ──
  Meta-prompt: ~1098 tokens
  ✓ Strategy: This prompt utilizes a direct, task-oriented approach. The system prompt establishes a clear role for the model (geother
  Parse rate: 100%  |  MSE: 0.0007

Best design for gemma-3-12b-it: round 0, parse=100%, mse=0.0007

phi-3.5-mini-instruct:
  Strategy: The system message is designed to be clear and instructive, guiding the model on the task at hand. The user template includes placeholders for the fea
  Parse: 100%  MSE: 0.0013  (designed in round 9)

phi-mini-moe-instruct:
  Strategy: The system message content and tone are professional and informati

In [12]:
# ── Full benchmark with auto-designed prompts ──────────────────────
# Same sample test set used in Step 7 for fair comparison

agentic_performance = []
agentic_full_results = {}

for endpoint_cfg in MODEL_ENDPOINTS:
    label = endpoint_cfg['label']
    design = agentic_results[label]['design']
    if design is None:
        print(f"Skipping {label} — no valid auto-designed prompt")
        continue
    sys_p = design['system_prompt']
    usr_t = design['user_template']

    records = []
    for _, row in test_subset_large.iterrows():
        record = pred_auto(row, endpoint_cfg, sys_p, usr_t)
        records.append(record)

    df_res = pd.DataFrame(records)
    agentic_full_results[label] = df_res

    # Save per-model results
    df_res.to_csv(OUT_DIR / f"agentic_benchmark_{label}.csv", index=False)

    # Evaluate same as Step 7
    perf = evaluate(df_res, f"{label} (auto)")
    agentic_performance += perf

df_agentic_perf = pd.DataFrame(agentic_performance)
df_agentic_perf.to_csv(OUT_DIR / "agentic_benchmark_performance.csv", index=False)
display(df_agentic_perf)


[phi-3.5-mini-instruct (auto)] Horizon +15m -> MSE_LLM: 0.0003 | MSE_ML: 0.0067 | R2_LLM: 0.9818 | R2_ML: 0.4854
[phi-3.5-mini-instruct (auto)] Horizon +60m -> MSE_LLM: 0.0004 | MSE_ML: 0.0060 | R2_LLM: 0.9743 | R2_ML: 0.5756
[phi-3.5-mini-instruct (auto)] Horizon +240m -> MSE_LLM: 0.0020 | MSE_ML: 0.0051 | R2_LLM: 0.8626 | R2_ML: 0.6313
[phi-3.5-mini-instruct (auto)] Horizon +1440m -> MSE_LLM: 0.0104 | MSE_ML: 0.0050 | R2_LLM: 0.2579 | R2_ML: 0.6163
[phi-mini-moe-instruct (auto)] Horizon +15m -> MSE_LLM: 0.0002 | MSE_ML: 0.0067 | R2_LLM: 0.9821 | R2_ML: 0.4854
[phi-mini-moe-instruct (auto)] Horizon +60m -> MSE_LLM: 0.0003 | MSE_ML: 0.0060 | R2_LLM: 0.9819 | R2_ML: 0.5756
[phi-mini-moe-instruct (auto)] Horizon +240m -> MSE_LLM: 0.0007 | MSE_ML: 0.0051 | R2_LLM: 0.9502 | R2_ML: 0.6313
[phi-mini-moe-instruct (auto)] Horizon +1440m -> MSE_LLM: 0.0018 | MSE_ML: 0.0050 | R2_LLM: 0.8657 | R2_ML: 0.6163
[gemma-3-12b-it (auto)] Horizon +15m -> MSE_LLM: 0.0004 | MSE_ML: 0.0067 | R2_LLM: 0.9706 

,Model,Horizon,MSE_LLM,MSE_ML,R2_LLM,R2_ML
0,phi-3.5-mini-instruct (auto),15,0.000253,0.006733,0.981825,0.485427
1,phi-3.5-mini-instruct (auto),60,0.000386,0.005992,0.974280,0.575629
2,phi-3.5-mini-instruct (auto),240,0.002022,0.005103,0.862636,0.631258
3,phi-3.5-mini-instruct (auto),1440,0.010380,0.005045,0.257909,0.616294
4,phi-mini-moe-instruct (auto),15,0.000236,0.006733,0.982108,0.485427
5,phi-mini-moe-instruct (auto),60,0.000258,0.005992,0.981934,0.575629
6,phi-mini-moe-instruct (auto),240,0.000697,0.005103,0.950153,0.631258
7,phi-mini-moe-instruct (auto),1440,0.001784,0.005045,0.865697,0.616294
8,gemma-3-12b-it (auto),15,0.000384,0.006733,0.970624,0.485427
9,gemma-3-12b-it (auto),60,0.000658,0.005992,0.953393,0.575629


In [13]:
# ── Save auto-designed prompts and round history ───────────────────

for label, res in agentic_results.items():
    design = res['design']
    history = res['history']

    # Save the final prompt design
    prompt_path = PROMPT_DIR / f"auto_prompt_{label}.json"
    save_prompts(design, prompt_path)

    # Save the full round history (prompt evolution log)
    history_path = OUT_DIR / f"agentic_round_history_{label}.json"
    # Clean non-serialisable values before saving
    clean_history = []
    for h in history:
        ch = {k: (v if not isinstance(v, float) or np.isfinite(v) else None)
              for k, v in h.items()}
        clean_history.append(ch)
    save_prompts(clean_history, history_path)

print("\nAll auto-designed prompts and histories saved ✓")


Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/auto_prompt_phi-3.5-mini-instruct.json  (1.2 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Results/agentic_round_history_phi-3.5-mini-instruct.json  (13.7 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/auto_prompt_phi-mini-moe-instruct.json  (1.3 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Results/agentic_round_history_phi-mini-moe-instruct.json  (7.6 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Prompts/auto_prompt_gemma-3-12b-it.json  (2.4 KB)
Saved → /home/jupyter-xue.li/ByteMe_week_4/Results/agentic_round_history_gemma-3-12b-it.json  (24.4 KB)

All auto-designed prompts and histories saved ✓


In [14]:
# ── Comparison: Manual prompt vs Auto-designed prompt vs ML ────────

# Merge Step 7 (manual) and Step 9 (auto) performance tables
df_manual = df_performance_metrics.copy()
df_manual = df_manual.rename(columns={
    'MSE_LLM': 'MSE_Manual', 'R2_LLM': 'R2_Manual',
})

df_auto = df_agentic_perf.copy()
df_auto['Model'] = df_auto['Model'].str.replace(' (auto)', '', regex=False)
df_auto = df_auto.rename(columns={
    'MSE_LLM': 'MSE_Auto', 'R2_LLM': 'R2_Auto',
})

comparison = pd.merge(
    df_manual[['Model', 'Horizon', 'MSE_Manual', 'R2_Manual', 'MSE_ML', 'R2_ML']],
    df_auto[['Model', 'Horizon', 'MSE_Auto', 'R2_Auto']],
    on=['Model', 'Horizon'],
    how='outer',
)

# Who wins per row?
comparison['Best_MSE'] = comparison[['MSE_Manual', 'MSE_Auto', 'MSE_ML']].idxmin(axis=1)
comparison['Best_MSE'] = comparison['Best_MSE'].map({
    'MSE_Manual': 'Manual LLM', 'MSE_Auto': 'Auto LLM', 'MSE_ML': 'ML (XGB)',
})

print("═" * 80)
print("COMPARISON: Manual Prompt vs Auto-Designed Prompt vs ML Baseline")
print("═" * 80)
display(comparison)

# Summary: how often each method wins
win_counts = comparison['Best_MSE'].value_counts()
print("\nWins by method (lowest MSE per model×horizon):")
display(win_counts)

# Report draft
print("\n── Agentic report draft ──")
for label, res in agentic_results.items():
    d = res['design']
    if d is None:
        print(f"\n{label}: No valid auto-designed prompt produced")
        continue
    print(f"\n{label}:")
    print(f"  Strategy chosen by agent: {d['strategy_notes']}")
    print(f"  Designed in round {d['round']}, test parse rate: {d['parse_rate']:.0%}")


════════════════════════════════════════════════════════════════════════════════
COMPARISON: Manual Prompt vs Auto-Designed Prompt vs ML Baseline
════════════════════════════════════════════════════════════════════════════════


,Model,Horizon,MSE_Manual,R2_Manual,MSE_ML,R2_ML,MSE_Auto,R2_Auto,Best_MSE
0,gemma-3-12b-it,15,0.000193,0.985270,0.006733,0.485427,0.000384,0.970624,Manual LLM
1,gemma-3-12b-it,60,0.000203,0.985649,0.005992,0.575629,0.000658,0.953393,Manual LLM
2,gemma-3-12b-it,240,0.000366,0.973567,0.005103,0.631258,0.001998,0.855600,Manual LLM
3,gemma-3-12b-it,1440,0.001654,0.874240,0.005045,0.616294,0.004899,0.627443,Manual LLM
4,phi-3.5-mini-instruct,15,0.000253,0.980683,0.006733,0.485427,0.000253,0.981825,Manual LLM
5,phi-3.5-mini-instruct,60,0.000197,0.986027,0.005992,0.575629,0.000386,0.974280,Manual LLM
6,phi-3.5-mini-instruct,240,0.000269,0.980573,0.005103,0.631258,0.002022,0.862636,Manual LLM
7,phi-3.5-mini-instruct,1440,0.001464,0.888645,0.005045,0.616294,0.010380,0.257909,Manual LLM
8,phi-mini-moe-instruct,15,0.000247,0.981157,0.006733,0.485427,0.000236,0.982108,Auto LLM
9,phi-mini-moe-instruct,60,0.000244,0.982738,0.005992,0.575629,0.000258,0.981934,Manual LLM



Wins by method (lowest MSE per model×horizon):


Best_MSE
Manual LLM    11
Auto LLM       1
Name: count, dtype: int64


── Agentic report draft ──

phi-3.5-mini-instruct:
  Strategy chosen by agent: The system message is designed to be clear and instructive, guiding the model on the task at hand. The user template includes placeholders for the features JSON, which will be replaced with actual data during model inference. The JSON schema is specified to ensure the output is in the correct format. Few-shot examples are not included in this prompt to maintain focus on the core task and avoid overwhelming the model with too much data. The prompt is kept concise to fit within the token limit and to ensure clarity.
  Designed in round 9, test parse rate: 100%

phi-mini-moe-instruct:
  Strategy chosen by agent: The system message content and tone are professional and informative. Few-shot examples are included to provide context and guidance. The input/output formatting style is compact and tabular, with clear instructions on the desired output format. The prompting technique used is chain-of-thought, as the 

## Step 9: Draft Report

What this does: programmatically generates a full Week 4 report from computed metrics across all pipeline stages.

Why it matters: data-driven reporting ensures the narrative matches actual results and is reproducible.

Principle: never hard-code metric values in reports — pull them from variables so the report auto-updates when you re-run.

In [15]:
# ═══════════════════════════════════════════════════════════════════
# Auto-generated Week 4 Report
# ═══════════════════════════════════════════════════════════════════

def _section(title):
    w = 72
    print(f"\n{'─'*w}")
    print(f"  {title}")
    print(f"{'─'*w}\n")

# ── 1. Approach ────────────────────────────────────────────────────
_section("1. APPROACH")

model_names = ", ".join(cfg['label'] for cfg in MODEL_ENDPOINTS)
n_versions = len(VERSIONS)
n_features = len(FEATURES_FULL)

print(
    f"We evaluated {len(MODEL_ENDPOINTS)} LLM endpoints ({model_names}) on a "
    f"geothermal temperature-delta forecasting task with {len(HORIZONS)} prediction "
    f"horizons (+{', +'.join(str(h) for h in HORIZONS)} min).\n"
    f"\nEach model received {n_features} sensor-derived features and was asked to "
    f"return a strict JSON response with four target predictions and a confidence score.\n"
    f"\nWe tested {n_versions} prompt versions (zero-shot and few-shot with "
    f"{'verbose and compact'} formatting styles) on a small {N_TEST_SAMPLES}-sample "
    f"subset, selected the best version per model by parse reliability then MSE, and "
    f"ran a full benchmark on {N_TEST_LARGE} evenly-spaced test rows.\n"
    f"\nAs an ML baseline, Multi-output XGBoost predictions from Week 3 were "
    f"evaluated on the same test rows."
)

# ── 2. Prompt Selection ───────────────────────────────────────────
_section("2. PROMPT SELECTION (Best Version per Model)")

for _, row in best_by_model.iterrows():
    model_summary = summary[
        (summary['model_label'] == row['model_label']) &
        (summary['version'] == row['version'])
    ].iloc[0]
    print(
        f"  {row['model_label']:30s} -> {row['version']:20s}  "
        f"parse={model_summary['parse_success_rate']:.0%}  "
        f"MSE={model_summary['mse_overall']:.4f}"
    )
print(f"\n  Overall best prompt version: {overall_best_version}")

# ── 3. Error Analysis ─────────────────────────────────────────────
_section("3. ERROR ANALYSIS")

failures = results[~results['parse_ok']].copy()
n_total = len(results)
n_fail = len(failures)
print(f"Total queries: {n_total}  |  Failures: {n_fail}  |  Failure rate: {n_fail/n_total:.1%}\n")

if n_fail > 0:
    # Breakdown by error type
    print("  Failure breakdown by error type:")
    err_counts = failures['parse_error'].value_counts()
    for err_type, count in err_counts.items():
        print(f"    {err_type:20s}: {count:3d}  ({count/n_total:.1%} of all queries)")

    # Breakdown by model
    print("\n  Failure breakdown by model:")
    model_fail = failures.groupby('model_label').size()
    model_total = results.groupby('model_label').size()
    for model in model_total.index:
        f_count = model_fail.get(model, 0)
        t_count = model_total[model]
        print(f"    {model:30s}: {f_count:3d}/{t_count:3d}  ({f_count/t_count:.0%})")

    # Sample failure responses
    print("\n  Sample failure responses (up to 3):")
    for i, (_, fail_row) in enumerate(failures.head(3).iterrows()):
        raw_snip = str(fail_row['raw_response'])[:150].replace('\n', ' ')
        print(f"    [{fail_row['model_label']}] {fail_row['parse_error']}: {raw_snip}")
else:
    print("  No parse failures detected across any prompt version or model.")

print(
    f"\nCommon failure modes include malformed JSON, missing target keys, "
    f"and out-of-range predictions (valid range: {VALID_RANGE}). "
    f"Prompt wording and formatting style affected compliance differently by model."
)

# ── 4. Full Benchmark: Manual Prompts vs ML ────────────────────────
_section("4. FULL BENCHMARK — Manual Prompt vs ML (XGB)")

for model in df_performance_metrics['Model'].unique():
    df_m = df_performance_metrics[df_performance_metrics['Model'] == model]
    avg_mse_llm = df_m['MSE_LLM'].mean()
    avg_mse_ml = df_m['MSE_ML'].mean()
    avg_r2_llm = df_m['R2_LLM'].mean()
    avg_r2_ml = df_m['R2_ML'].mean()
    winner = "LLM" if avg_mse_llm < avg_mse_ml else "ML (XGB)"
    print(f"  {model}:")
    print(f"    Avg MSE  — LLM: {avg_mse_llm:.4f}  |  ML: {avg_mse_ml:.4f}  -> Winner: {winner}")
    print(f"    Avg R²   — LLM: {avg_r2_llm:.4f}  |  ML: {avg_r2_ml:.4f}")
    for _, hr in df_m.iterrows():
        print(f"      +{int(hr['Horizon']):>4d}m  MSE_LLM={hr['MSE_LLM']:.4f}  MSE_ML={hr['MSE_ML']:.4f}  "
              f"R2_LLM={hr['R2_LLM']:.4f}  R2_ML={hr['R2_ML']:.4f}")

# ── 5. Agentic Prompt Auto-Design ─────────────────────────────────
_section("5. AGENTIC PROMPT AUTO-DESIGN")

for label, res in agentic_results.items():
    d = res['design']
    hist = res['history']
    if d is None:
        print(f"  {label}: No valid auto-designed prompt produced across {AGENTIC_MAX_ROUNDS} rounds.")
        continue
    n_ok_rounds = sum(1 for h in hist if h.get('status') == 'ok')
    print(f"  {label}:")
    print(f"    Best round:    {d['round']} / {AGENTIC_MAX_ROUNDS-1}")
    print(f"    Parse rate:    {d['parse_rate']:.0%}")
    print(f"    MSE:           {d['mse']:.4f}")
    print(f"    Rounds tried:  {len(hist)} ({n_ok_rounds} valid designs)")
    print(f"    Strategy:      {d['strategy_notes']}")

    # Show round-by-round evolution
    for h in hist:
        status = h.get('status', '?')
        if status == 'ok':
            print(f"      Round {h['round']}: parse={h['parse_rate']:.0%}  MSE={h['mse']:.4f}")
        else:
            err = h.get('error', 'unknown')
            print(f"      Round {h['round']}: {status} — {err[:80]}")

# ── 6. Final Comparison: Manual vs Auto vs ML ─────────────────────
_section("6. FINAL COMPARISON — Manual vs Auto-Designed vs ML")

display(comparison[['Model', 'Horizon', 'MSE_Manual', 'MSE_Auto', 'MSE_ML', 'Best_MSE']])

print(f"\nWins by method (lowest MSE per model × horizon):")
for method, count in win_counts.items():
    total = len(comparison)
    print(f"  {method:15s}: {count}/{total}  ({count/total:.0%})")

# ── 7. Conclusions ─────────────────────────────────────────────────
_section("7. CONCLUSIONS")

top_method = win_counts.idxmax()
top_wins = win_counts.max()
total_matchups = len(comparison)

# Average MSE across all models and horizons
avg_manual = comparison['MSE_Manual'].mean()
avg_auto = comparison['MSE_Auto'].mean()
avg_ml = comparison['MSE_ML'].mean()
best_overall = min(avg_manual, avg_auto, avg_ml)
best_label = {avg_manual: "Manual LLM prompts", avg_auto: "Auto-designed LLM prompts",
              avg_ml: "ML (XGB) baseline"}[best_overall]

print(
    f"Across {total_matchups} model × horizon combinations, {top_method} achieved the "
    f"lowest MSE in {top_wins}/{total_matchups} cases ({top_wins/total_matchups:.0%}).\n"
    f"\nAverage MSE across all models and horizons:"
    f"\n  Manual LLM:        {avg_manual:.4f}"
    f"\n  Auto-designed LLM: {avg_auto:.4f}"
    f"\n  ML (XGB):          {avg_ml:.4f}"
    f"\n\nOverall best average MSE: {best_label} ({best_overall:.4f}).\n"
    f"\nKey findings:"
    f"\n  - Prompt format (verbose vs compact) and few-shot count significantly affect "
    f"parse reliability across models."
    f"\n  - The agentic auto-design loop produced model-specific prompt strategies that "
    f"differ from manually crafted templates."
    f"\n  - Parse failures remain the primary bottleneck for LLM-based forecasting — "
    f"improving JSON compliance is critical for practical deployment."
    f"\n  - Longer prediction horizons (240 min, 1440 min) are generally harder for "
    f"all methods, reflecting increased forecast uncertainty."
)

print(f"\n{'═'*72}")
print(f"  END OF REPORT")
print(f"{'═'*72}")


────────────────────────────────────────────────────────────────────────
  1. APPROACH
────────────────────────────────────────────────────────────────────────

We evaluated 3 LLM endpoints (phi-3.5-mini-instruct, phi-mini-moe-instruct, gemma-3-12b-it) on a geothermal temperature-delta forecasting task with 4 prediction horizons (+15, +60, +240, +1440 min).

Each model received 15 sensor-derived features and was asked to return a strict JSON response with four target predictions and a confidence score.

We tested 9 prompt versions (zero-shot and few-shot with verbose and compact formatting styles) on a small 5-sample subset, selected the best version per model by parse reliability then MSE, and ran a full benchmark on 100 evenly-spaced test rows.

As an ML baseline, Multi-output XGBoost predictions from Week 3 were evaluated on the same test rows.

────────────────────────────────────────────────────────────────────────
  2. PROMPT SELECTION (Best Version per Model)
──────────────────

,Model,Horizon,MSE_Manual,MSE_Auto,MSE_ML,Best_MSE
0,gemma-3-12b-it,15,0.000193,0.000384,0.006733,Manual LLM
1,gemma-3-12b-it,60,0.000203,0.000658,0.005992,Manual LLM
2,gemma-3-12b-it,240,0.000366,0.001998,0.005103,Manual LLM
3,gemma-3-12b-it,1440,0.001654,0.004899,0.005045,Manual LLM
4,phi-3.5-mini-instruct,15,0.000253,0.000253,0.006733,Manual LLM
5,phi-3.5-mini-instruct,60,0.000197,0.000386,0.005992,Manual LLM
6,phi-3.5-mini-instruct,240,0.000269,0.002022,0.005103,Manual LLM
7,phi-3.5-mini-instruct,1440,0.001464,0.010380,0.005045,Manual LLM
8,phi-mini-moe-instruct,15,0.000247,0.000236,0.006733,Auto LLM
9,phi-mini-moe-instruct,60,0.000244,0.000258,0.005992,Manual LLM



Wins by method (lowest MSE per model × horizon):
  Manual LLM     : 11/12  (92%)
  Auto LLM       : 1/12  (8%)

────────────────────────────────────────────────────────────────────────
  7. CONCLUSIONS
────────────────────────────────────────────────────────────────────────

Across 12 model × horizon combinations, Manual LLM achieved the lowest MSE in 11/12 cases (92%).

Average MSE across all models and horizons:
  Manual LLM:        0.0006
  Auto-designed LLM: 0.0020
  ML (XGB):          0.0057

Overall best average MSE: Manual LLM prompts (0.0006).

Key findings:
  - Prompt format (verbose vs compact) and few-shot count significantly affect parse reliability across models.
  - The agentic auto-design loop produced model-specific prompt strategies that differ from manually crafted templates.
  - Parse failures remain the primary bottleneck for LLM-based forecasting — improving JSON compliance is critical for practical deployment.
  - Longer prediction horizons (240 min, 1440 min) ar